# FVAA — Fase sin baterías

**Pregunta:** ¿cuánto mejora la predicción de energía solar al añadir meteorología al histórico de producción y al calendario?

Objetivos: (1) siguiente hora, (2) perfil horario de 24 horas, (3) total de las próximas 24 horas. Todos se emiten en `t`, sin utilizar observaciones posteriores a `t`.

Niveles: 1. Producción + calendario; 2. Añadir irradiación histórica; 3. Añadir pronósticos archivados publicados antes de `t` (solo cuando haya datos válidos).

Se utilizan cortes temporales de entrenamiento, validación y prueba. Las pruebas de 2020 ya se han observado en experimentos anteriores: sus resultados son **retrospectivos, no una evaluación independiente nueva**.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from fvaa.produccion import cargar_produccion, agregar_produccion_horaria
from fvaa.irradiacion import cargar_irradiacion, agregar_irradiacion_15min, agregar_irradiacion_horaria
from fvaa.features import crear_variables_historicas, anadir_irradiacion_historica

ANIOS = [2020]  # Ampliar cuando estén disponibles los CSV de otros años.
INICIO_VAL = pd.Timestamp("2020-09-01", tz="UTC")
INICIO_TEST = pd.Timestamp("2020-11-01", tz="UTC")
RAW = ROOT / "data/raw/PTProsumer/PR20"
from fvaa.pronosticos import cargar_gfs, alinear_gfs

PRONOSTICOS = ROOT / "data/interim/gfs_pr20_2020/gfs_madeira_3h_2020_36h.csv"
if not PRONOSTICOS.is_file():
    PRONOSTICOS = ROOT / "data/interim/gfs_pr20_2020/gfs_madeira_3h_2020.csv"
LATENCIA_GFS_H = 6  # Supuesto de trabajo, no disponibilidad certificada.


## 1. Preparación de datos a una hora

Producción: potencia media en W durante una hora, equivalente a `P_media/1000` kWh. Irradiación: 60 observaciones en Wh/m²; se suma y se expresa como irradiancia media horaria en W/m². **Los timestamps de irradiación marcan el final del intervalo**, por eso el paso final a horas usa el extremo derecho.

Se mantiene una rejilla horaria completa para que `shift(24)` signifique exactamente un día atrás.

In [ ]:
# PRODUCCIÓN
produccion = []

for anio in ANIOS:
    nombre = f"pr20_prod_{anio}.csv"
    ruta = RAW / nombre

    if ruta.is_dir():
        ruta = ruta / nombre

    datos = cargar_produccion(ruta)
    horas = agregar_produccion_horaria(datos)
    del datos  # Liberar las lecturas por segundo tras agregarlas.

    indice = pd.date_range(
        f"{anio}-01-01", f"{anio}-12-31 23:00",
        freq="h", tz="UTC"
    )
    produccion.append(horas.reindex(indice))

produccion = pd.concat(produccion).sort_index().asfreq("h")


# IRRADIACIÓN
archivos = []

for anio in range(min(ANIOS), max(ANIOS) + 2):
    nombre = f"pr20_irrad_{anio}.csv"
    ruta = RAW / nombre

    if ruta.is_dir():
        ruta = ruta / nombre if (ruta / nombre).is_file() else ruta / f"pr20_rad_{anio}.csv"

    if ruta.is_file():
        archivos.append(cargar_irradiacion(ruta))

if not archivos:
    raise FileNotFoundError("No se encontraron archivos de irradiación")

minutos = pd.concat(archivos, ignore_index=True)
irradiacion = agregar_irradiacion_horaria(
    agregar_irradiacion_15min(minutos)
)


# COMPROBACIÓN
print("Horas de producción:", len(produccion))
print("Horas válidas:", produccion["energia_estimada_kwh"].notna().sum())
print("Horas con irradiación:", irradiacion["GHI_media_w_m2"].notna().sum())


## 2. Variables de entrada y tres objetivos

En una emisión `t`, la fila `h00` representa energía de `[t,t+1h)` y `h23` representa `[t+23h,t+24h)`. El total de 24 h es la suma de esas 24 horas, únicamente si todas son válidas.

Los retardos de producción y de irradiación se calculan sobre la rejilla completa y usan horas anteriores a `t`. La irradiación horaria etiqueta al inicio la suma de minutos cuyo timestamp marca su final. Así, el retardo de 1 h acaba en `t`; no incorpora observaciones futuras. Esto supone disponibilidad al terminar la hora: CAMS retrospectivo no acredita esa disponibilidad real y el nivel 2 debe interpretarse bajo ese supuesto.


In [ ]:

base = crear_variables_historicas(produccion)
X = anadir_irradiacion_historica(base, irradiacion).drop(columns="objetivo_kwh")

energia = produccion["energia_estimada_kwh"]
perfil = pd.DataFrame({f"h{h}": energia.shift(-h) for h in range(24)})

objetivos = {
    "1h": perfil[["h0"]],
    "perfil_24h": perfil,
    "total_24h": perfil.sum(axis=1, min_count=24).to_frame("total_kwh")
}

nivel1 = [
    "energia_anterior_1h", "energia_anterior_2h",
    "energia_anterior_24h", "hora", "mes"
]

meteo = [
    "ghi_anterior_1h", "ghi_anterior_24h",
    "ghi_despejado_anterior_1h", "ghi_despejado_anterior_24h"
]


# Referencias disponibles en t: únicamente horas ya terminadas.
referencias = {
    "1h": energia.shift(24).to_frame("h0"),
    "perfil_24h": pd.DataFrame({
        f"h{i}": energia.shift(24 - i) for i in range(24)
    }),
    "total_24h": energia.shift(1).rolling(24).sum().to_frame("total_kwh")
}


## 3. Pronósticos históricos GFS

Usamos el CSV procesado de GFS 2020 para el punto aproximado de Madeira (32,75° N, 17° O). DSWRF es una media de tres horas en W/m²; se repite en las tres horas y los valores negativos se truncan a cero. La transformación de los GRIB originales debe documentarse y verificarse por separado.

Para cada emisión t y horizonte h seleccionamos el último pronóstico de la hora t+h cuya disponibilidad supuesta sea anterior o igual a t. Se estudia la cobertura con latencias de 0, 3 y 6 horas. La latencia principal de 6 horas es una hipótesis, no una prueba de publicación efectiva. CAMS histórico también conserva su limitación de disponibilidad.

Si el archivo solo contiene plazos hasta 24 h, las 24 horas completas no son evaluables con latencia positiva. No se imputan pronósticos faltantes ni se usan ejecuciones futuras.
Los objetivos de 24 h usan siempre las 24 horas de pronóstico. Si no hay cobertura completa en las tres particiones, esos objetivos no se evalúan con GFS y el notebook lo indica; no se usan horizontes parciales.


In [ ]:
if PRONOSTICOS.is_file():
    gfs = cargar_gfs(PRONOSTICOS)
    print("Negativos truncados a cero:", (gfs.dswrf_media_w_m2 < 0).sum())
    for latencia in [0, 3, 6]:
        pronostico = alinear_gfs(X.index, gfs, latencia)
        print(f"Latencia {latencia} h: 1 h = {pronostico.iloc[:, 0].count()}, "
              f"24 h = {pronostico.notna().all(axis=1).sum()}")
    X = X.join(alinear_gfs(X.index, gfs, LATENCIA_GFS_H))
else:
    print("Nivel 3 pendiente: falta el CSV GFS")

## 4. Entrenamiento y comparación

Para comparar niveles se mantienen **las mismas fechas y objetivos evaluados**. `t+24 h <= corte` impide que objetivos de entrenamiento o validación invadan el periodo siguiente. La validación permite elegir el candidato con menor MAE, **sin decidir mirando el test**. La tabla también informa todos los modelos para comparar la misma familia con y sin meteorología. Ridge usa un escalado calculado solo a partir del entrenamiento.

La tabla conserva resultados de todos los modelos, con MAE, MSE y RMSE. La regresión de perfil usa 24 salidas simultáneas: Ridge y RF las admiten directamente; GB se ajusta mediante 24 regresores independientes.

In [ ]:
def crear_modelos(n):
    gb = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)

    return {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1)),
        "RF": RandomForestRegressor(
            n_estimators=100, min_samples_leaf=5, random_state=42, n_jobs=4
        ),
        "GB": MultiOutputRegressor(gb) if n > 1 else gb
    }


def errores(y, pred):
    mse = mean_squared_error(np.ravel(y), np.ravel(pred))
    return {
        "MAE": mean_absolute_error(np.ravel(y), np.ravel(pred)),
        "MSE": mse,
        "RMSE": np.sqrt(mse)
    }


def dividir(y, h, columnas):
    validos = (
        X[columnas].notna().all(axis=1)
        & y.notna().all(axis=1)
        & referencias["1h" if h == 1 else "perfil_24h"].notna().all(axis=1)
    )
    fin = X.index + pd.Timedelta(hours=h)

    train = validos & (fin <= INICIO_VAL)
    val = validos & (X.index >= INICIO_VAL) & (fin <= INICIO_TEST)
    test = validos & (X.index >= INICIO_TEST)

    return train, val, test


niveles_base = {
    "Sin meteorología": nivel1,
    "Con irradiación": nivel1 + meteo
}

estudios = ["historica"]
if PRONOSTICOS.is_file():
    estudios.append("pronosticos")

# Los objetivos de 24 h exigen los pronósticos de las 24 horas siguientes. Si las tres
# particiones no tienen cobertura suficiente, NO se evalúan con menos horas ni se rellenan.
cobertura_24h = False
if PRONOSTICOS.is_file():
    columnas = nivel1 + meteo + [f"pronostico_h{i}" for i in range(24)]
    cobertura_24h = min(m.sum() for m in dividir(objetivos["perfil_24h"], 24, columnas)) >= 30
    print("Cobertura GFS suficiente para objetivos de 24 h:", cobertura_24h)

resultados = []

for objetivo, y in objetivos.items():
    h = 1 if objetivo == "1h" else 24

    for estudio in estudios:
        niveles = niveles_base.copy()

        if estudio == "pronosticos":
            if h == 24 and not cobertura_24h:
                print(f"No evaluado: {objetivo} con GFS (faltan pronósticos de 24 h con la latencia supuesta)")
                continue
            pronosticos = [f"pronostico_h{i}" for i in range(h)]
            niveles["3A Pronóstico sin CAMS"] = nivel1 + pronosticos
            niveles["3B CAMS + pronóstico"] = nivel1 + meteo + pronosticos

        comunes = list(set().union(*niveles.values()))
        train, val, test = dividir(y, h, comunes)

        if min(train.sum(), val.sum(), test.sum()) < 30:
            print(f"Datos insuficientes: {objetivo}, {estudio} (train={train.sum()}, val={val.sum()}, test={test.sum()})")
            continue

        for nivel, columnas in niveles.items():
            for nombre, modelo in crear_modelos(y.shape[1]).items():

                modelo.fit(X.loc[train, columnas], y.loc[train].squeeze("columns"))

                pred_val = np.maximum(modelo.predict(X.loc[val, columnas]), 0)
                mae_val = errores(y.loc[val], pred_val)["MAE"]

                modelo.fit(
                    X.loc[train | val, columnas],
                    y.loc[train | val].squeeze("columns")
                )

                pred = np.maximum(modelo.predict(X.loc[test, columnas]), 0)

                resultados.append({
                    "estudio": estudio,
                    "objetivo": objetivo,
                    "nivel": nivel,
                    "horas_pronostico": h if nivel.startswith("3") else 0,
                    "modelo": nombre,
                    "MAE_validacion": mae_val,
                    "latencia_gfs_h": LATENCIA_GFS_H if nivel.startswith("3") else None,
                    "n_train": int(train.sum()),
                    "n_val": int(val.sum()),
                    "horas_test": int(test.sum()),
                    **errores(y.loc[test], pred)
                })

resultados = pd.DataFrame(resultados)
display(resultados.round(5))

seleccionados = resultados.loc[
    resultados.groupby(["estudio", "objetivo", "nivel"])["MAE_validacion"].idxmin()
]

display(seleccionados.round(5))


### Comparación emparejada y referencias

La ganancia meteorológica se calcula **para el mismo modelo y el mismo objetivo**: `100 × (MAE_sin − MAE_con) / MAE_sin`. Para 24 h añadimos la referencia de repetir el perfil del día anterior, que solo utiliza información pasada.

La máscara común se aplica antes de entrenar y evaluar. Para 24 h exige también las 24 horas anteriores completas, necesarias para la referencia.


In [ ]:
# MEJORA AL AÑADIR METEOROLOGÍA

tabla = resultados[resultados["estudio"] == "historica"].pivot_table(
    index=["objetivo", "modelo"], columns="nivel", values="MAE"
)

tabla["mejora_meteo_pct"] = 100 * (
    1 - tabla["Con irradiación"] / tabla["Sin meteorología"]
)

display(tabla.round(4))

# REFERENCIAS SOBRE LAS MISMAS MUESTRAS QUE LOS MODELOS
filas = []
for objetivo, pred in referencias.items():
    y = objetivos[objetivo]
    h = 1 if objetivo == "1h" else 24
    _, _, test = dividir(y, h, nivel1 + meteo)
    assert pred.loc[test].notna().all().all()
    assert resultados.loc[
        (resultados["estudio"] == "historica")
        & (resultados["objetivo"] == objetivo), "horas_test"
    ].eq(test.sum()).all()
    filas.append({"objetivo": objetivo, "horas_test": int(test.sum()),
                  **errores(y.loc[test], pred.loc[test])})

_, _, test = dividir(objetivos["1h"], 1, nivel1 + meteo)
filas.append({"objetivo": "1h, última hora", "horas_test": int(test.sum()),
              **errores(objetivos["1h"].loc[test], energia.shift(1).loc[test])})
display(pd.DataFrame(filas).round(5))


# Comparación adicional de los tres niveles sobre la intersección GFS.
resultados_gfs = resultados[resultados["estudio"] == "pronosticos"]
if not resultados_gfs.empty:
    tabla_gfs = resultados_gfs.pivot_table(
        index=["objetivo", "modelo"], columns="nivel", values="MAE"
    )
    # Mejora (%) = 100 * (MAE_antes - MAE_despues) / MAE_antes, mismo modelo y objetivo.
    tabla_gfs["mejora_nivel2_vs_1_pct"] = 100 * (1 - tabla_gfs["Con irradiación"] / tabla_gfs["Sin meteorología"])
    tabla_gfs["mejora_3A_vs_1_pct"] = 100 * (1 - tabla_gfs["3A Pronóstico sin CAMS"] / tabla_gfs["Sin meteorología"])
    tabla_gfs["mejora_3B_vs_2_pct"] = 100 * (1 - tabla_gfs["3B CAMS + pronóstico"] / tabla_gfs["Con irradiación"])
    tabla_gfs["mejora_3B_vs_3A_pct"] = 100 * (1 - tabla_gfs["3B CAMS + pronóstico"] / tabla_gfs["3A Pronóstico sin CAMS"])
    tabla_gfs["horas_pronostico"] = [1 if obj == "1h" else 24 for obj, _ in tabla_gfs.index]
    display(tabla_gfs.round(5))
    # Muestras comunes a las cuatro configuraciones, por objetivo y partición.
    display(resultados_gfs.groupby("objetivo")[["n_train", "n_val", "horas_test"]].first())
    filas_gfs = []
    for objetivo in resultados_gfs["objetivo"].unique():
        y = objetivos[objetivo]
        h = 1 if objetivo == "1h" else 24
        columnas = nivel1 + meteo + [f"pronostico_h{i}" for i in range(h)]  # unión de los cuatro niveles
        _, _, test = dividir(y, h, columnas)
        assert resultados_gfs.loc[resultados_gfs["objetivo"] == objetivo, "horas_test"].eq(test.sum()).all()
        filas_gfs.append({"objetivo": objetivo, "referencia": "día anterior",
                          "horas_test": int(test.sum()),
                          **errores(y.loc[test], referencias[objetivo].loc[test])})
        if h == 1:
            filas_gfs.append({"objetivo": objetivo, "referencia": "última hora",
                              "horas_test": int(test.sum()),
                              **errores(y.loc[test], energia.shift(1).loc[test])})
    display(pd.DataFrame(filas_gfs).round(5))


## 5. Incertidumbre: cuantiles P10, P50 y P90

Para objetivos escalares (1 h y total 24 h) entrenamos Gradient Boosting con pérdida de cuantiles. El intervalo P10–P90 tiene una cobertura *nominal* del 80 %, que debe compararse con la cobertura observada. Ordenar cuantiles cruzados **no calibra el intervalo**.

In [ ]:
incertidumbre = []

for objetivo in ["1h", "total_24h"]:
    y = objetivos[objetivo]
    h = 1 if objetivo == "1h" else 24
    for estudio in estudios:
        niveles = niveles_base.copy()
        if estudio == "pronosticos":
            if h == 24 and not cobertura_24h:
                continue
            pron = [f"pronostico_h{i}" for i in range(h)]
            niveles["3A Pronóstico sin CAMS"] = nivel1 + pron
            niveles["3B CAMS + pronóstico"] = nivel1 + meteo + pron
        comunes = list(set().union(*niveles.values()))
        train, val, test = dividir(y, h, comunes)
        if min(train.sum(), val.sum(), test.sum()) < 30:
            continue
        for nivel, columnas in niveles.items():
            predicciones = []
            for q in [0.1, 0.5, 0.9]:
                modelo = GradientBoostingRegressor(
                    loss="quantile", alpha=q, n_estimators=100,
                    max_depth=3, random_state=42
                )
                modelo.fit(X.loc[train | val, columnas], y.loc[train | val].iloc[:, 0])
                predicciones.append(np.maximum(modelo.predict(X.loc[test, columnas]), 0))
            q10, q50, q90 = np.sort(predicciones, axis=0)
            real = y.loc[test].iloc[:, 0]
            incertidumbre.append({
                "estudio": estudio, "objetivo": objetivo, "nivel": nivel,
                "horas_pronostico": h if nivel.startswith("3") else 0,
                "horas_test": int(test.sum()),
                "cobertura_pct": 100 * np.mean((real >= q10) & (real <= q90)),
                "anchura_media_kwh": np.mean(q90 - q10),
                "MAE_mediana_kwh": mean_absolute_error(real, q50)
            })

display(pd.DataFrame(incertidumbre).round(4))


## 6. Escala de entrenamiento

Comparamos el mismo RF y las mismas entradas sobre las mismas fechas de **validación**, variando el histórico de entrenamiento (90 días, 180 días o todo el disponible antes de septiembre). No se optimiza mirando test. No equivale a demostrar escalado entre instalaciones distintas.

In [ ]:
y = objetivos["1h"]
columnas = nivel1 + meteo
tr, va, _ = dividir(y, 1, columnas)

modelo = RandomForestRegressor(
    n_estimators=100, min_samples_leaf=5, random_state=42, n_jobs=4
)

escala = []

for dias in [90, 180, None]:
    train = tr if dias is None else tr & (
        X.index >= INICIO_VAL - pd.Timedelta(days=dias)
    )

    if train.sum() < 30:
        continue

    modelo.fit(X.loc[train, columnas], y.loc[train].iloc[:, 0])
    pred = modelo.predict(X.loc[va, columnas]).clip(min=0)

    escala.append({
        "entrenamiento": "todo" if dias is None else f"{dias} días",
        "n_train": int(train.sum()),
        "MAE_validacion": mean_absolute_error(y.loc[va].iloc[:, 0], pred)
    })

display(pd.DataFrame(escala).round(5))

print("Tamaño con menor MAE de validación:", pd.DataFrame(escala).loc[pd.DataFrame(escala)["MAE_validacion"].idxmin(), "entrenamiento"])


## 7. Resultados y conclusiones para la tutoría

Una gráfica por objetivo representa el MAE de la comparación emparejada. Interpretar mejoras y empeoramientos por modelo; no seleccionar el ganador según el test.

In [ ]:
for nombre_objetivo in tabla.index.get_level_values("objetivo").unique():
    ax = tabla.loc[nombre_objetivo, ["Sin meteorología", "Con irradiación"]].plot.bar(
        figsize=(7, 4), title=f"MAE test, solo niveles 1 y 2 (histórico) — {nombre_objetivo}"
    )
    ax.set_ylabel("MAE (kWh)")
    ax.set_xlabel("Modelo")
    plt.tight_layout()
    plt.show()

print("Recordatorio: el test de 2020 es conocido; CAMS no acredita disponibilidad inmediata.")
print("El nivel 3 requiere disponibilidad acreditada y cobertura suficiente para los tres objetivos.")

if not resultados_gfs.empty:
    for objetivo in tabla_gfs.index.get_level_values("objetivo").unique():
        ax = tabla_gfs.loc[objetivo, ["Sin meteorología", "Con irradiación", "3A Pronóstico sin CAMS", "3B CAMS + pronóstico"]].plot.bar(
            figsize=(7, 4), title=f"MAE test — {objetivo} (GFS {1 if objetivo == '1h' else 24} h, latencia {LATENCIA_GFS_H} h)"
        )
        ax.set_ylabel("MAE (kWh)")
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))  # fuera de los ejes: no tapa barras
        plt.tight_layout()
        plt.show()
